# AURA VTO — Temporary Authenticated GPU Service (Kaggle)

This notebook prepares and runs the AURA GPU Virtual Try-On FastAPI service inside a **Kaggle GPU Notebook session** (using Kaggle's free GPU: **T4 x2** or **P100**).

### Operational Guarantees & Constraints
- **Zero Commercial APIs**: Completely self-contained open-source model execution (FASHN VTON v1.5 + DWPose). Zero calls to OpenAI, Replicate, Fal, or hosted FASHN.
- **Zero Weight Modification**: Model weights are frozen and verified against SHA-256 signatures.
- **Private Supabase Storage**: Inputs and outputs are stored exclusively in private Supabase buckets (`vto_inputs`, `vto_results`) using owner-only RLS policies.
- **Zero Secrets in Code**: Credentials (`SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`, `VTO_JWT_SECRET`) must be provided via Kaggle User Secrets (`kaggle_secrets.UserSecretsClient`) or runtime input, never committed.
- **Temporary & Quota-Limited**: Kaggle is ephemeral (30h/week quota, max 12h session duration). This notebook is for temporary development and integration testing only.

In [ ]:
# Step 1: Environment Detection & Verified Repository Setup
import os
import sys
import platform
import shutil
import subprocess
from pathlib import Path

IN_KAGGLE = bool(os.getenv("KAGGLE_KERNEL_RUN_TYPE"))
print(f"Kaggle Environment Detected: {IN_KAGGLE}")
print(f"Python: {platform.python_version()} on {platform.system()} ({platform.machine()})")

def is_genuine_aura_repo(p: Path) -> bool:
    """Verifies that path p contains the genuine AURA repository with DWPose."""
    if not p.is_dir():
        return False
    has_wholebody = (p / "services" / "vto" / "fashn" / "dwpose" / "wholebody.py").exists()
    has_pipeline = (p / "services" / "vto" / "fashn" / "decoupled_pipeline.py").exists()
    return has_wholebody and has_pipeline

# 1. Candidate paths to search for existing genuine AURA repository
candidate_paths = [
    Path.cwd(),
    Path("/kaggle/working/aura"),
    Path("/kaggle/working/auramain"),
    Path("/kaggle/working"),
]
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").glob("*"):
        if p.is_dir():
            candidate_paths.append(p)
            for sub in p.glob("*"):
                if sub.is_dir():
                    candidate_paths.append(sub)

AURA_ROOT = None
for candidate in candidate_paths:
    if is_genuine_aura_repo(candidate):
        AURA_ROOT = candidate.resolve()
        break

# 2. In Kaggle, if repository is missing or incomplete (e.g. wholebody.py absent), perform clean clone
if IN_KAGGLE and AURA_ROOT is None:
    target_repo = Path("/kaggle/working/aura")
    print(f"Genuine repository not found in candidates. Preparing clean checkout at {target_repo}...")
    
    # Preserve existing downloaded weights if present
    weights_src = target_repo / "services" / "vto" / "weights"
    backup_weights = Path("/kaggle/working/_weights_backup")
    if weights_src.exists() and any(weights_src.iterdir()):
        print("Preserving existing downloaded weights before checkout...")
        shutil.rmtree(str(backup_weights), ignore_errors=True)
        shutil.move(str(weights_src), str(backup_weights))

    # Remove incomplete directory skeleton so git clone can succeed
    shutil.rmtree(str(target_repo), ignore_errors=True)

    # Clone genuine repository
    print("Cloning AURA repository from https://github.com/Shriyash05/auramain.git...")
    try:
        clone_res = subprocess.run(
            ["git", "clone", "https://github.com/Shriyash05/auramain.git", str(target_repo)],
            capture_output=True,
            text=True
        )
        if clone_res.returncode == 0 and is_genuine_aura_repo(target_repo):
            AURA_ROOT = target_repo.resolve()
            print(f"Successfully checked out genuine repository to: {AURA_ROOT}")
        else:
            print(f"Git clone output: {clone_res.stdout}\n{clone_res.stderr}")
    except Exception as e:
        print(f"Clone attempt failed: {e}")

    # Restore preserved weights into checkout
    if backup_weights.exists():
        dest_weights = target_repo / "services" / "vto" / "weights"
        dest_weights.parent.mkdir(parents=True, exist_ok=True)
        if dest_weights.exists():
            shutil.rmtree(str(dest_weights), ignore_errors=True)
        shutil.move(str(backup_weights), str(dest_weights))
        print("Preserved weights successfully restored to checkout.")

# 3. Enforce genuine checkout
if AURA_ROOT is None or not is_genuine_aura_repo(AURA_ROOT):
    raise SystemExit(
        f"BLOCKED: Could not establish a genuine AURA repository checkout with DWPose!\n"
        f"wholebody.py was not found in services/vto/fashn/dwpose/ at any candidate path.\n"
        "Action required: Ensure https://github.com/Shriyash05/auramain.git is cloned into /kaggle/working/aura."
    )

print(f"Verified AURA Repository Root: {AURA_ROOT}")
if str(AURA_ROOT) not in sys.path:
    sys.path.insert(0, str(AURA_ROOT))

# 4. Ensure package __init__.py files exist
(AURA_ROOT / "services" / "__init__.py").touch(exist_ok=True)
(AURA_ROOT / "services" / "vto" / "fashn" / "__init__.py").touch(exist_ok=True)

# 5. Ensure compatibility alias services/vto_gpu/fashn/dwpose/wholebody.py
gpu_dwpose_dir = AURA_ROOT / "services" / "vto_gpu" / "fashn" / "dwpose"
gpu_dwpose_dir.mkdir(parents=True, exist_ok=True)
(AURA_ROOT / "services" / "vto_gpu" / "fashn" / "__init__.py").write_text(
    'from services.vto.fashn.decoupled_pipeline import DecoupledTryOnPipeline\n'
    'from services.vto.fashn.tryon_mmdit import TryOnModel\n'
    '__all__ = ["DecoupledTryOnPipeline", "TryOnModel"]\n',
    encoding="utf-8"
)
(gpu_dwpose_dir / "__init__.py").write_text(
    'from services.vto.fashn.dwpose.dwpose import DWposeDetector, draw_pose\n'
    'from services.vto.fashn.dwpose.wholebody import Wholebody\n'
    '__all__ = ["DWposeDetector", "draw_pose", "Wholebody"]\n',
    encoding="utf-8"
)
(gpu_dwpose_dir / "wholebody.py").write_text(
    'from services.vto.fashn.dwpose.wholebody import Wholebody\n'
    '__all__ = ["Wholebody"]\n',
    encoding="utf-8"
)

# 6. Ensure deployment validator is present in AURA_ROOT/tools
tools_dir = AURA_ROOT / "tools"
tools_dir.mkdir(parents=True, exist_ok=True)
validator_file = tools_dir / "validate_vto_deployment.py"
if not validator_file.exists():
    # Bootstrap canonical fail-closed deployment validator if missing from commit
    validator_file.write_text(
        '"""Safe deployment preflight validator."""\n'
        'from __future__ import annotations\n'
        'import os, re, sys\n'
        'from pathlib import Path\n'
        'from urllib.parse import urlparse\n'
        'ROOT = Path(sys.argv[1]).resolve() if len(sys.argv) > 1 and Path(sys.argv[1]).is_dir() else Path(__file__).resolve().parents[1]\n'
        'SERVER_ONLY = ("SUPABASE_SERVICE_ROLE_KEY", "VTO_JWT_SECRET")\n'
        'REQUIRED = ("SUPABASE_URL", *SERVER_ONLY, "VTO_WEIGHTS_DIR", "VTO_MODEL_RESOLUTION", "VTO_INPUT_BUCKET", "VTO_OUTPUT_BUCKET")\n'
        'def fail(m): print(f"FAIL: {m}")\n'
        'def main():\n'
        '    ok = True\n'
        '    for name in REQUIRED:\n'
        '        present = bool(os.getenv(name))\n'
        '        print(f"{\'OK\' if present else \'FAIL\'}: {name} {\'is set\' if present else \'is missing\'}")\n'
        '        ok &= present\n'
        '    url = os.getenv("SUPABASE_URL", "")\n'
        '    if url and (urlparse(url).scheme != "https" or not urlparse(url).netloc): fail("SUPABASE_URL must be HTTPS"); ok = False\n'
        '    res = os.getenv("VTO_MODEL_RESOLUTION", "")\n'
        '    if res and not re.fullmatch(r"\\d{2,4},\\d{2,4}", res): fail("VTO_MODEL_RESOLUTION format invalid"); ok = False\n'
        '    w = os.getenv("VTO_WEIGHTS_DIR", "")\n'
        '    if w and not Path(w).is_dir(): fail("VTO_WEIGHTS_DIR does not exist"); ok = False\n'
        '    for n in ("VTO_INPUT_BUCKET", "VTO_OUTPUT_BUCKET"):\n'
        '        if not os.getenv(n, "").strip(): fail(f"{n} must be non-empty"); ok = False\n'
        '    findings = []\n'
        '    for p in (ROOT / ".env", ROOT / ".env.example"):\n'
        '        if p.exists():\n'
        '            t = p.read_text(encoding="utf-8", errors="ignore")\n'
        '            for k in SERVER_ONLY:\n'
        '                if re.search(rf"^\\s*(?:EXPO_PUBLIC_)?{k}\\s*=\\s*[^#\\s]", t, re.M): findings.append(f"{p.name}:{k}")\n'
        '            if re.search(r"https://[^\\s]*(?:ngrok|trycloudflare|loca\\.lt)", t, re.I): findings.append(f"{p.name}:tunnel")\n'
        '    if findings:\n'
        '        for f in findings: fail(f"Policy violation: {f}")\n'
        '        ok = False\n'
        '    else: print("OK: No secret/tunnel violations in env files")\n'
        '    return 0 if ok else 1\n'
        'if __name__ == "__main__": sys.exit(main())\n',
        encoding="utf-8"
    )

# 7. Ensure services/vto_gpu files exist
gpu_srv_dir = AURA_ROOT / "services" / "vto_gpu"
gpu_srv_dir.mkdir(parents=True, exist_ok=True)
if not (gpu_srv_dir / "config.py").exists():
    (gpu_srv_dir / "config.py").write_text('''import os\nfrom pathlib import Path\nfrom pydantic import BaseModel, Field\nclass Settings(BaseModel):\n    supabase_url: str = Field(pattern=r"^https://[A-Za-z0-9.-]+\\.supabase\\.co$")\n    supabase_service_role_key: str = Field(min_length=32)\n    jwt_secret: str = Field(min_length=16)\n    weights_dir: Path\n    model_resolution: tuple[int, int] = (672, 432)\n    input_bucket: str = "vto_inputs"\n    output_bucket: str = "vto_results"\n    max_concurrent_jobs: int = 1\n    @classmethod\n    def from_env(cls) -> "Settings":\n        res = os.getenv("VTO_MODEL_RESOLUTION", "672,432")\n        h, w = (int(x.strip()) for x in res.split(","))\n        return cls(supabase_url=os.environ["SUPABASE_URL"], supabase_service_role_key=os.environ["SUPABASE_SERVICE_ROLE_KEY"], jwt_secret=os.environ["VTO_JWT_SECRET"], weights_dir=Path(os.environ["VTO_WEIGHTS_DIR"]), model_resolution=(h, w), input_bucket=os.getenv("VTO_INPUT_BUCKET", "vto_inputs"), output_bucket=os.getenv("VTO_OUTPUT_BUCKET", "vto_results"), max_concurrent_jobs=int(os.getenv("VTO_MAX_CONCURRENT_JOBS", "1")))\n''', encoding="utf-8")
# Ensure services/vto_gpu/app.py is always up to date with full model loading & lifespan
    (gpu_srv_dir / "app.py").write_text('''"""Run with: uvicorn services.vto_gpu.app:app --host 127.0.0.1 --port 8001."""\nimport asyncio, io, os, tempfile, time, traceback, uuid\nfrom contextlib import asynccontextmanager\nfrom pathlib import Path\nfrom fastapi import Depends, FastAPI, Header, HTTPException, status\nfrom jose import JWTError, jwt\nfrom PIL import Image\nfrom pydantic import BaseModel, Field\nfrom supabase import create_client\nfrom .config import Settings\n\nclass JobCreate(BaseModel):\n    category: str = Field(pattern="^(tops|bottoms)$")\n    person_input_storage_key: str = Field(pattern=r"^[A-Za-z0-9_./-]+$")\n    garment_input_storage_key: str = Field(pattern=r"^[A-Za-z0-9_./-]+$")\n    garment_id: str = Field(min_length=1, max_length=128)\n    outfit_name: str | None = Field(default=None, max_length=100)\n    idempotency_key: str = Field(min_length=16, max_length=128)\n\nclass Service:\n    def __init__(self, settings: Settings):\n        self.settings=settings; self.db=create_client(settings.supabase_url, settings.supabase_service_role_key)\n        self.pipeline=None; self.lock=asyncio.Semaphore(settings.max_concurrent_jobs)\n    def user(self, token: str) -> str:\n        try: return jwt.decode(token, self.settings.jwt_secret, algorithms=["HS256"], audience="authenticated")["sub"]\n        except (JWTError, KeyError): raise HTTPException(401, "Invalid authentication token")\n    def load(self):\n        if self.pipeline is None:\n            print("[VTO SERVICE] Loading DecoupledTryOnPipeline (MMDiT + DWPose)...", flush=True)\n            try:\n                from services.vto.fashn.decoupled_pipeline import DecoupledTryOnPipeline\n                pipeline = DecoupledTryOnPipeline(self.settings.weights_dir, input_shape=self.settings.model_resolution)\n                self.pipeline = pipeline\n                print("[VTO SERVICE] >>> Readiness flag set to true (both MMDiT and DWPose loaded successfully) <<<", flush=True)\n            except Exception:\n                print(f"[VTO SERVICE ERROR] Failed to load DecoupledTryOnPipeline:\\n{traceback.format_exc()}", flush=True)\n                raise\n    def _owned_garment(self, user_id: str, garment_id: str):\n        data=self.db.table("garments").select("id").eq("id",garment_id).eq("user_id",user_id).execute().data\n        if not data: raise HTTPException(403,"Garment is not accessible")\n    async def process(self, job_id: str):\n        async with self.lock:\n            row=self.db.table("vto_jobs").select("*").eq("id",job_id).single().execute().data\n            if row["status"] == "cancelled": return\n            self.db.table("vto_jobs").update({"status":"processing","started_at":"now()"}).eq("id",job_id).execute()\n            started=time.monotonic()\n            try:\n                self.load()\n                # Storage keys, never client paths; files are isolated and removed automatically.\n                with tempfile.TemporaryDirectory(prefix="aura-vto-") as temp:\n                    p=Path(temp); person=p/"person"; garment=p/"garment"; output=p/"result.png"\n                    person.write_bytes(self.db.storage.from_(self.settings.input_bucket).download(row["person_input_storage_key"]))\n                    garment.write_bytes(self.db.storage.from_(self.settings.input_bucket).download(row["garment_input_storage_key"]))\n                    a,b=Image.open(person).convert("RGB"),Image.open(garment).convert("RGB")\n                    result=self.pipeline(a,b,category=row["category"],garment_photo_type="flat-lay",num_samples=1,num_timesteps=30,guidance_scale=1.5,seed=42)\n                    image=result.images[0]; image.save(output,"PNG")\n                    if self.db.table("vto_jobs").select("status").eq("id",job_id).single().execute().data["status"] == "cancelled": return\n                    key=f"{row['user_id']}/{job_id}.png"; self.db.storage.from_(self.settings.output_bucket).upload(key, output.read_bytes(), {"content-type":"image/png","upsert":"false"})\n                self.db.table("vto_jobs").update({"status":"completed","output_storage_key":key,"completed_at":"now()","processing_duration_ms":round((time.monotonic()-started)*1000)}).eq("id",job_id).execute()\n            except Exception:\n                self.db.table("vto_jobs").update({"status":"failed","error_code":"INFERENCE_FAILED","safe_error_message":"VTO processing failed.","failed_at":"now()"}).eq("id",job_id).execute()\n\nsettings: Settings | None = None; service: Service | None = None\n@asynccontextmanager\nasync def lifespan(app: FastAPI):\n    global settings, service\n    print("[VTO LIFESPAN] >>> Starting AURA GPU VTO lifespan initialization <<<", flush=True)\n    try:\n        settings = Settings.from_env()\n        service = Service(settings)\n        print(f"[VTO LIFESPAN] Settings loaded. Starting model loading in thread pool...", flush=True)\n        await asyncio.to_thread(service.load)\n        print("[VTO LIFESPAN] >>> Lifespan startup completed: Service is READY. /ready will return true. <<<", flush=True)\n    except Exception:\n        print(f"[VTO LIFESPAN FATAL ERROR] Model initialization failed during lifespan startup:\\n{traceback.format_exc()}", flush=True)\n        raise\n    yield\n    print("[VTO LIFESPAN] Lifespan shutdown: Cleaning up service...", flush=True)\n    service = None\n    settings = None\napp=FastAPI(title="AURA GPU VTO", lifespan=lifespan)\nfrom fastapi.middleware.cors import CORSMiddleware\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=True,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\ndef current_user(authorization: str | None = Header(default=None)) -> str:\n    if not authorization or not authorization.startswith("Bearer "): raise HTTPException(401,"Authentication required")\n    return service.user(authorization[7:])\n@app.get("/health")\ndef health(): return {"status":"ok","message":"VTO GPU service active and ready."}\n@app.get("/ready")\ndef ready(): return {"ready": service is not None and service.pipeline is not None}\n@app.get("/v1/vto/diagnostics/auth")\ndef auth_diagnostics(authorization: str | None = Header(default=None)):\n    """Safe, sanitized authentication diagnostic endpoint.\n    Exposes NO secrets, NO tokens, NO personal claims, NO file paths, and NO bucket names.\n    Returns safe structural diagnostic codes to help identify why POST /v1/vto/jobs returns 401.\n    """\n    secret_configured = bool(service and service.settings and service.settings.jwt_secret)\n    if not authorization:\n        return {\n            "status": "fail",\n            "code": "AUTH_HEADER_MISSING",\n            "message": "Authorization header is missing or was stripped by proxy/tunnel.",\n            "header_present": False,\n            "scheme_valid": False,\n            "token_structure_valid": False,\n            "jwt_secret_configured": secret_configured,\n        }\n    if not authorization.startswith("Bearer "):\n        return {\n            "status": "fail",\n            "code": "AUTH_SCHEME_INVALID",\n            "message": "Authorization scheme must be 'Bearer <token>'.",\n            "header_present": True,\n            "scheme_valid": False,\n            "token_structure_valid": False,\n            "jwt_secret_configured": secret_configured,\n        }\n    token = authorization[7:].strip()\n    parts = token.split(".")\n    if len(parts) != 3:\n        return {\n            "status": "fail",\n            "code": "TOKEN_STRUCTURE_INVALID",\n            "message": "Token does not have 3 dot-separated JWT segments.",\n            "header_present": True,\n            "scheme_valid": True,\n            "token_structure_valid": False,\n            "jwt_secret_configured": secret_configured,\n        }\n    try:\n        from jose import jwt as jose_jwt\n        unverified_header = jose_jwt.get_unverified_header(token)\n        unverified_claims = jose_jwt.get_unverified_claims(token)\n    except Exception:\n        return {\n            "status": "fail",\n            "code": "TOKEN_PARSE_ERROR",\n            "message": "Failed to decode unverified JWT structure.",\n            "header_present": True,\n            "scheme_valid": True,\n            "token_structure_valid": False,\n            "jwt_secret_configured": secret_configured,\n        }\n    token_alg = unverified_header.get("alg")\n    has_sub = "sub" in unverified_claims\n    aud = unverified_claims.get("aud")\n    is_aud_authenticated = (aud == "authenticated")\n    exp = unverified_claims.get("exp")\n    is_expired = (exp is not None and exp < time.time())\n\n    if token_alg != "HS256":\n        return {\n            "status": "fail",\n            "code": "ALGORITHM_MISMATCH",\n            "message": f"Token algorithm '{token_alg}' is not HS256.",\n            "header_present": True,\n            "scheme_valid": True,\n            "token_structure_valid": True,\n            "algorithm": token_alg,\n            "has_sub": has_sub,\n            "is_aud_authenticated": is_aud_authenticated,\n            "is_expired": is_expired,\n            "jwt_secret_configured": secret_configured,\n        }\n    if not is_aud_authenticated:\n        return {\n            "status": "fail",\n            "code": "AUDIENCE_MISMATCH",\n            "message": f"Token audience is not 'authenticated' (found: '{aud}'). Anon keys cannot authenticate user jobs.",\n            "header_present": True,\n            "scheme_valid": True,\n            "token_structure_valid": True,\n            "algorithm": token_alg,\n            "has_sub": has_sub,\n            "is_aud_authenticated": False,\n            "is_expired": is_expired,\n            "jwt_secret_configured": secret_configured,\n        }\n    if not has_sub:\n        return {\n            "status": "fail",\n            "code": "MISSING_SUB_CLAIM",\n            "message": "Token is missing 'sub' claim. A user access token is required, not an anon key.",\n            "header_present": True,\n            "scheme_valid": True,\n            "token_structure_valid": True,\n            "algorithm": token_alg,\n            "has_sub": False,\n            "is_aud_authenticated": is_aud_authenticated,\n            "is_expired": is_expired,\n            "jwt_secret_configured": secret_configured,\n        }\n    if is_expired:\n        return {\n            "status": "fail",\n            "code": "TOKEN_EXPIRED",\n            "message": "Token has expired. Client must refresh the Supabase session.",\n            "header_present": True,\n            "scheme_valid": True,\n            "token_structure_valid": True,\n            "algorithm": token_alg,\n            "has_sub": has_sub,\n            "is_aud_authenticated": is_aud_authenticated,\n            "is_expired": True,\n            "jwt_secret_configured": secret_configured,\n        }\n    try:\n        jose_jwt.decode(token, service.settings.jwt_secret, algorithms=["HS256"], audience="authenticated")\n        return {\n            "status": "ok",\n            "code": "AUTH_SUCCESS",\n            "message": "Token is structurally valid, signed correctly, and authorized for VTO jobs.",\n            "header_present": True,\n            "scheme_valid": True,\n            "token_structure_valid": True,\n            "algorithm": token_alg,\n            "has_sub": True,\n            "is_aud_authenticated": True,\n            "is_expired": False,\n            "signature_valid": True,\n            "jwt_secret_configured": True,\n        }\n    except Exception:\n        return {\n            "status": "fail",\n            "code": "SIGNATURE_VERIFICATION_FAILED",\n            "message": "Cryptographic signature verification failed. VTO_JWT_SECRET in server environment does not match the secret that signed this token.",\n            "header_present": True,\n            "scheme_valid": True,\n            "token_structure_valid": True,\n            "algorithm": token_alg,\n            "has_sub": True,\n            "is_aud_authenticated": True,\n            "is_expired": False,\n            "signature_valid": False,\n            "jwt_secret_configured": True,\n        }\n\n@app.post("/v1/vto/jobs", status_code=202)\nasync def create_job(body: JobCreate, user_id: str=Depends(current_user)):\n    service._owned_garment(user_id,body.garment_id)\n    existing=service.db.table("vto_jobs").select("id,status").eq("user_id",user_id).eq("idempotency_key",body.idempotency_key).execute().data\n    if existing:return existing[0]\n    job_id="vto_"+uuid.uuid4().hex; data=body.model_dump()|{"id":job_id,"user_id":user_id,"status":"queued","model_version":"fashn-vton-1.5","model_resolution":list(settings.model_resolution),"inference_parameters":{"num_timesteps":30,"guidance_scale":1.5,"seed":42}}\n    service.db.table("vto_jobs").insert(data).execute(); asyncio.create_task(service.process(job_id)); return {"id":job_id,"status":"queued"}\n@app.get("/v1/vto/jobs/{job_id}")\ndef job(job_id:str,user_id:str=Depends(current_user)):\n    row=service.db.table("vto_jobs").select("*").eq("id",job_id).eq("user_id",user_id).single().execute().data\n    if not row: raise HTTPException(404,"Job not found")\n    if row.get("output_storage_key"): row["result_signed_url"]=service.db.storage.from_(settings.output_bucket).create_signed_url(row["output_storage_key"],300)["signedURL"]\n    return row\n@app.post("/v1/vto/jobs/{job_id}/cancel")\ndef cancel(job_id:str,user_id:str=Depends(current_user)):\n    service.db.table("vto_jobs").update({"status":"cancelled","cancelled_at":"now()"}).eq("id",job_id).eq("user_id",user_id).eq("status","queued").execute(); return {"id":job_id,"status":"cancelled"}\n''', encoding="utf-8")

print("AURA repository verification and package configuration: SUCCESS.")


In [ ]:
# Step 2: GPU Verification & VRAM Check
import torch

print("--- GPU Verification ---")
if not torch.cuda.is_available():
    raise SystemExit("BLOCKED: No CUDA GPU available! Please enable GPU accelerator in Kaggle Notebook Settings (Settings -> Accelerator -> GPU T4 x2 or P100).")

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"GPU Available: True (Count: {gpu_count})")
print(f"Primary GPU: {gpu_name}")
print(f"Physical VRAM: {total_vram_gb:.2f} GB")

MIN_VRAM_REQUIRED_GB = 12.0
if total_vram_gb < MIN_VRAM_REQUIRED_GB:
    raise SystemExit(f"BLOCKED: GPU VRAM ({total_vram_gb:.2f} GB) is below required {MIN_VRAM_REQUIRED_GB} GB.")

print("Status: GPU hardware requirement VERIFIED.")


In [ ]:
# Step 3: Targeted AURA VTO Dependencies Installation & Verification
# Safeguards against torchvision circular import contamination and module shadowing.
# Preserves Kaggle preinstalled PyTorch without touching unrelated packages.
import os
import sys
import subprocess
from pathlib import Path

print("--- Step 3: AURA VTO Dependencies & Environment Verification ---")

# 1. Inspect for local module shadowing (torchvision.py, torchvision/, extension.py)
print("\n[1] Checking for Local Module Shadowing...")
shadow_candidates = ["torchvision.py", "torchvision", "extension.py"]
shadow_paths_checked = [Path.cwd(), AURA_ROOT]

detected_shadows = []
for base_dir in shadow_paths_checked:
    if base_dir.exists():
        for name in shadow_candidates:
            target = base_dir / name
            if target.exists():
                detected_shadows.append(target)

if detected_shadows:
    print(f"CRITICAL: Local file(s) shadowing 'torchvision' found: {detected_shadows}")
    raise SystemExit(
        f"BLOCKED: Module shadowing detected! Local files are obstructing torchvision:\n"
        + "\n".join(f"  - {p}" for p in detected_shadows)
    )
print("  OK: No local files (torchvision.py, torchvision/, extension.py) shadowing torchvision.")

# 2. Inspect active sys.modules for contaminated torchvision state
print("\n[2] Inspecting Active Kernel sys.modules for Contamination...")
tv_modules = [k for k in sys.modules if k == "torchvision" or k.startswith("torchvision.")]
if tv_modules:
    tv_mod = sys.modules.get("torchvision")
    if tv_mod is not None and not hasattr(tv_mod, "extension"):
        print(f"KERNEL CONTAMINATION DETECTED in active session: {tv_modules}")
        print("sys.modules['torchvision'] is partially initialized and missing '.extension'.")
        raise SystemExit(
            "BLOCKED: Kaggle notebook kernel is contaminated from an earlier interrupted run.\n"
            "Unsafe in-process reloading cannot repair partially initialized C++ extension bindings.\n"
            "RECOVERY ACTION REQUIRED:\n"
            "  1. In the Kaggle notebook menu, click 'Kernel' -> 'Restart' (or 'Session' -> 'Restart Session').\n"
            "  2. Run notebook cells sequentially from Step 1.\n"
            "  3. Do not re-run dependency cells repeatedly in the same contaminated kernel session."
        )
    else:
        print(f"  torchvision currently present in sys.modules: {tv_modules}")
else:
    print("  OK: No prior torchvision entries in sys.modules (clean kernel state).")

# 3. Clean Subprocess Verification of Torch and Torchvision
# Must run in a clean subprocess to distinguish genuine installation failure from kernel contamination
print("\n[3] Clean Subprocess Verification of PyTorch & Torchvision...")
sub_test_cmd = [
    sys.executable, "-c",
    "import sys\n"
    "import torch\n"
    "print(f'  Clean Subprocess torch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')\n"
    "if torch.cuda.is_available():\n"
    "    print(f'  Clean Subprocess Primary GPU: {torch.cuda.get_device_name(0)}')\n"
    "import torchvision\n"
    "print(f'  Clean Subprocess torchvision: {torchvision.__version__}')\n"
    "assert hasattr(torchvision, 'extension'), 'torchvision missing extension attribute'\n"
]

sub_res = subprocess.run(sub_test_cmd, capture_output=True, text=True)
print(sub_res.stdout.strip())
if sub_res.returncode != 0:
    print(f"FAIL: Clean subprocess verification failed (Exit Code {sub_res.returncode}):")
    if sub_res.stderr:
        print(sub_res.stderr.strip())
    raise SystemExit(
        "BLOCKED: Clean-subprocess torchvision import failed! Genuine installation problem detected.\n"
        "PyTorch and torchvision C++ extension bindings are broken in this environment."
    )
print("  Status: Clean subprocess PyTorch and Torchvision import VERIFIED.")

# 4. Targeted Installation of Required AURA VTO Dependencies
# DO NOT reinstall, downgrade, or upgrade torch or torchvision.
# Pin onnxruntime-gpu==1.20.2 as verified for CUDA 12.8 / Tesla T4.
print("\n[4] Installing Targeted AURA VTO Packages via subprocess.check_call...")
REQUIRED_AURA_PACKAGES = [
    "onnxruntime-gpu==1.20.2",
    "safetensors>=0.5.0",
    "huggingface_hub>=0.23.0",
    "einops>=0.8.0",
    "opencv-python-headless>=4.8.0",
    "fastapi>=0.110.0",
    "uvicorn[standard]>=0.28.0",
    "python-jose[cryptography]>=3.3.0",
    "supabase>=2.3.0",
]

install_cmd = [
    sys.executable, "-m", "pip", "install",
    "--no-warn-conflicts",
    *REQUIRED_AURA_PACKAGES
]

try:
    subprocess.check_call(install_cmd)
    print("  Targeted package installation completed successfully.")
except subprocess.CalledProcessError as e:
    # Non-fatal warnings with unrelated preinstalled Kaggle packages (e.g. bigframes, datasets, google-colab)
    print(f"  Note: pip reported non-zero code {e.returncode} due to unrelated environment warnings.")

# 5. In-Process Core Import Verification
print("\n[5] In-Process AURA VTO Core Imports Verification...")
aura_import_failures = []

def check_import(name: str):
    try:
        mod = __import__(name)
        ver = getattr(mod, "__version__", "available")
        print(f"  OK: {name:20s} (version: {ver})")
        return mod
    except Exception as err:
        print(f"  FAIL: {name:18s} -> {err}")
        aura_import_failures.append(f"{name}: {err}")
        return None

# Import torch first, then torchvision, then the remaining stack
torch_mod = check_import("torch")
tv_mod = check_import("torchvision")
ort_mod = check_import("onnxruntime")
cv2_mod = check_import("cv2")
np_mod = check_import("numpy")
pil_mod = check_import("PIL")
einops_mod = check_import("einops")
safetensors_mod = check_import("safetensors")
hf_mod = check_import("huggingface_hub")
fastapi_mod = check_import("fastapi")
uvicorn_mod = check_import("uvicorn")
jose_mod = check_import("jose")
supabase_mod = check_import("supabase")

if aura_import_failures:
    raise SystemExit("BLOCKED: AURA VTO core dependency import failure(s):\n  " + "\n  ".join(aura_import_failures))

# 6. Hardware & CUDAExecutionProvider Verification
print("\n[6] Hardware & ONNX Provider Verification...")
if not torch_mod.cuda.is_available():
    raise SystemExit("BLOCKED: PyTorch CUDA is not available! Enable GPU accelerator in Kaggle settings.")
print(f"  PyTorch CUDA: Available ({torch_mod.cuda.get_device_name(0)})")

providers = ort_mod.get_available_providers()
print(f"  ONNX Providers: {providers}")
if "CUDAExecutionProvider" not in providers:
    raise SystemExit("BLOCKED: onnxruntime-gpu does NOT have 'CUDAExecutionProvider' available!")
print("  ONNX CUDAExecutionProvider: VERIFIED AVAILABLE")

print("\n" + "=" * 60)
print("AURA VTO dependencies verified; unrelated Kaggle package conflicts may remain.")
print("=" * 60)


In [ ]:
# Step 4: Model & DWPose Weights Verification
import hashlib
from huggingface_hub import hf_hub_download

print("--- Model Weights Verification ---")
weights_dir = AURA_ROOT / "services" / "vto" / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
dwpose_dir = weights_dir / "dwpose"
dwpose_dir.mkdir(parents=True, exist_ok=True)

EXPECTED_ARTIFACTS = [
    {
        "path": weights_dir / "model.safetensors",
        "repo_id": "fashn-ai/fashn-vton-1.5",
        "revision": "7720683168567eb5a2a4c67f15116c6e29c83ded",
        "filename": "model.safetensors",
        "sha256": "d6cd38286885bc29fa487ea9383f80ffeb95862e7747c630d42c5d3c05bdd35a"
    },
    {
        "path": dwpose_dir / "yolox_l.onnx",
        "repo_id": "fashn-ai/DWPose",
        "revision": "548b5df25b84d9f4aac0611dfa1c2a7a12f15571",
        "filename": "yolox_l.onnx",
        "sha256": "7860ae79de6c89a3c1eb72ae9a2756c0ccfbe04b7791bb5880afabd97855a411"
    },
    {
        "path": dwpose_dir / "dw-ll_ucoco_384.onnx",
        "repo_id": "fashn-ai/DWPose",
        "revision": "548b5df25b84d9f4aac0611dfa1c2a7a12f15571",
        "filename": "dw-ll_ucoco_384.onnx",
        "sha256": "724f4ff2439ed61afb86fb8a1951ec39c6220682803b4a8bd4f598cd913b1843"
    }
]

def verify_file_sha256(file_path, expected_hash):
    if not file_path.exists():
        return False
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(8192 * 1024):
            h.update(chunk)
    return h.hexdigest() == expected_hash

for item in EXPECTED_ARTIFACTS:
    dest = item["path"]
    if verify_file_sha256(dest, item["sha256"]):
        print(f"Verified existing: {dest.name} (SHA-256 match)")
    else:
        print(f"Downloading verified artifact: {item['filename']} from {item['repo_id']}...")
        hf_hub_download(
            repo_id=item["repo_id"],
            filename=item["filename"],
            revision=item["revision"],
            local_dir=str(dest.parent),
            local_dir_use_symlinks=False
        )
        if not verify_file_sha256(dest, item["sha256"]):
            raise RuntimeError(f"SHA-256 mismatch after download for {dest.name}!")
        print(f"Downloaded and verified: {dest.name}")

print("All required model weights are present and verified.")


In [ ]:
# Step 4b: DWPose Architecture & Runtime Diagnostic
# Explicitly verifies wholebody.py, ONNX models, SHA-256 signatures, and imports.
import os
import sys
import hashlib
import subprocess
from pathlib import Path

print("=" * 60)
print("AURA VTO — DWPose Architecture & Runtime Diagnostic")
print("=" * 60)

# 1. Repository Commit SHA
commit_sha = "unknown"
try:
    git_res = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=str(AURA_ROOT),
        capture_output=True,
        text=True
    )
    if git_res.returncode == 0:
        commit_sha = git_res.stdout.strip()
    else:
        head_file = AURA_ROOT / ".git" / "HEAD"
        if head_file.exists():
            commit_sha = head_file.read_text().strip()[:40]
except Exception as e:
    commit_sha = f"error: {e}"
print(f"\n[1] Repository Commit SHA: {commit_sha}")

# 2. Discovered DWPose Source Path
dwpose_src_dir = AURA_ROOT / "services" / "vto" / "fashn" / "dwpose"
print(f"\n[2] DWPose Source Path: {dwpose_src_dir}")
if not dwpose_src_dir.exists():
    raise SystemExit(f"BLOCKED: DWPose source directory does not exist at {dwpose_src_dir}!")

# 3. DWPose-related Python Files
print("\n[3] DWPose-related Python Files:")
dwpose_py_files = sorted(list(dwpose_src_dir.glob("*.py")))
for pf in dwpose_py_files:
    print(f"  - {pf.name} ({pf.stat().st_size:,} bytes)")

wholebody_file = dwpose_src_dir / "wholebody.py"
if not wholebody_file.exists():
    raise SystemExit(f"BLOCKED: wholebody.py does not exist in {dwpose_src_dir}!")
print("  Status: wholebody.py VERIFIED PRESENT.")

# 4. DWPose Model Artifact Paths
weights_dwpose_dir = AURA_ROOT / "services" / "vto" / "weights" / "dwpose"
print(f"\n[4] DWPose Model Artifact Directory: {weights_dwpose_dir}")

EXPECTED_DWPOSE_ONNX = {
    "yolox_l.onnx": "7860ae79de6c89a3c1eb72ae9a2756c0ccfbe04b7791bb5880afabd97855a411",
    "dw-ll_ucoco_384.onnx": "724f4ff2439ed61afb86fb8a1951ec39c6220682803b4a8bd4f598cd913b1843",
}

# 5. SHA-256 Hashes of the DWPose ONNX Files
print("\n[5] SHA-256 Hashes of DWPose ONNX Artifacts:")
for onnx_name, expected_hash in EXPECTED_DWPOSE_ONNX.items():
    onnx_path = weights_dwpose_dir / onnx_name
    if not onnx_path.exists():
        raise SystemExit(f"BLOCKED: Required ONNX model missing: {onnx_path}")
    h = hashlib.sha256()
    with open(onnx_path, "rb") as f:
        while chunk := f.read(8192 * 1024):
            h.update(chunk)
    actual_hash = h.hexdigest()
    match = (actual_hash == expected_hash)
    print(f"  - {onnx_name}:")
    print(f"      Path:     {onnx_path}")
    print(f"      Size:     {onnx_path.stat().st_size:,} bytes")
    print(f"      SHA-256:  {actual_hash}")
    print(f"      Expected: {expected_hash}")
    print(f"      Match:    {'VERIFIED' if match else 'MISMATCH'}")
    if not match:
        raise SystemExit(f"BLOCKED: SHA-256 mismatch for {onnx_name}!")

# 6. Python Import Verification
print("\n[6] Python Import Verification:")
import importlib

test_modules = [
    "services.vto.fashn.dwpose.wholebody",
    "services.vto_gpu.fashn.dwpose.wholebody",
    "services.vto.fashn.dwpose",
    "services.vto_gpu.fashn.dwpose",
]
for mod_name in test_modules:
    try:
        mod = importlib.import_module(mod_name)
        wb_cls = getattr(mod, "Wholebody", None)
        print(f"  - import {mod_name}: OK (Wholebody class: {wb_cls is not None})")
    except Exception as e:
        print(f"  - import {mod_name}: FAIL -> {e}")
        raise SystemExit(f"BLOCKED: Import failed for {mod_name}: {e}")

from services.vto.fashn.dwpose.wholebody import Wholebody as CanonicalWholebody
from services.vto_gpu.fashn.dwpose.wholebody import Wholebody as AliasWholebody
assert CanonicalWholebody is AliasWholebody, "Canonical and alias Wholebody classes must be identical!"
print("  Canonical and alias Wholebody classes verified identical: OK")

# 7. Runtime Providers & Hardware Capabilities (Fail closed if CUDAExecutionProvider missing on GPU)
import onnxruntime as ort
import torch

print("\n[7] Runtime Providers & Hardware:")
providers = ort.get_available_providers()
print(f"  ONNX Execution Providers Available: {providers}")
has_cuda_ep = "CUDAExecutionProvider" in providers
print(f"  CUDAExecutionProvider Available:    {has_cuda_ep}")
print(f"  PyTorch CUDA Available:             {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Primary GPU:                        {torch.cuda.get_device_name(0)}")
    if not has_cuda_ep:
        raise SystemExit("BLOCKED: CUDA GPU detected but ONNX Runtime CUDAExecutionProvider is missing!")

# 8. Live Wholebody Smoke Test on Available Hardware
target_device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"\n[8] Running Wholebody Smoke Test on {target_device}...")
smoke_detector = CanonicalWholebody(checkpoints_dir=str(weights_dwpose_dir), device=target_device)
print("  ONNX InferenceSessions initialized successfully:")
print(f"    - session_det input:  {[i.name for i in smoke_detector.session_det.get_inputs()]}")
print(f"    - session_pose input: {[i.name for i in smoke_detector.session_pose.get_inputs()]}")

import numpy as np
dummy_input = np.zeros((512, 512, 3), dtype=np.uint8)
kps, scores = smoke_detector(dummy_input)
print(f"  Forward pass result: keypoints shape {kps.shape}, scores shape {scores.shape}")
del smoke_detector

print("\n" + "=" * 60)
print("DWPose component, artifacts, and runtime verification: VERIFIED.")
print("Overall status: AURA_VTO_BLOCKED (remains until live end-to-end inference verification).")
print("=" * 60)


In [ ]:
# Step 5: Server-Only Runtime Configuration & Preflight Validation
import os
import sys
import getpass
import subprocess
from pathlib import Path

print("--- Server-Only Runtime Configuration ---")

# Method A: Load from Kaggle Secrets (preferred)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for key in ["SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY", "VTO_JWT_SECRET"]:
        try:
            val = user_secrets.get_secret(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except Exception:
    pass

# Method B: Fallback to interactive input if not already set
for key in ["SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY", "VTO_JWT_SECRET"]:
    if not os.getenv(key):
        val = getpass.getpass(f"Enter {key} (hidden input): ").strip()
        if not val:
            raise SystemExit(f"BLOCKED: {key} is required for the GPU service to run.")
        os.environ[key] = val

# Set operational environment variables
os.environ["VTO_WEIGHTS_DIR"] = str(AURA_ROOT / "services" / "vto" / "weights")
os.environ.setdefault("VTO_MODEL_RESOLUTION", "672,432")
os.environ.setdefault("VTO_INPUT_BUCKET", "vto_inputs")
os.environ.setdefault("VTO_OUTPUT_BUCKET", "vto_results")

# Preflight Validation
print("\n--- Running Preflight Deployment Validation ---")
print(f"Detected AURA Repository Root: {AURA_ROOT}")

validator_path = AURA_ROOT / "tools" / "validate_vto_deployment.py"
print(f"Preflight Validator Path: {validator_path}")

if not validator_path.exists():
    raise SystemExit(
        f"ERROR: Preflight validator script does not exist at:\n  {validator_path}\n"
        f"Action required: Ensure 'tools/validate_vto_deployment.py' is present in the repository."
    )

result = subprocess.run(
    [sys.executable, str(validator_path), str(AURA_ROOT)],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print(result.stderr)
    raise SystemExit(f"Preflight validation failed (Exit Code {result.returncode}). Please review failures above.")

print("Server runtime configuration and preflight validation VERIFIED.")


In [ ]:
# Step 6: Start FastAPI Service via Controlled Process
import subprocess
import os
import sys
from pathlib import Path

print("--- Starting AURA GPU VTO Service ---")
log_file_path = Path("/kaggle/working/vto_service.log") if IN_KAGGLE else AURA_ROOT / "vto_service.log"
log_file = open(log_file_path, "w", encoding="utf-8")

cmd = [
    sys.executable, "-u", "-m", "uvicorn",
    "services.vto_gpu.app:app",
    "--host", "127.0.0.1",
    "--port", "8001"
]

server_env = os.environ.copy()
server_env["PYTHONUNBUFFERED"] = "1"

server_process = subprocess.Popen(
    cmd,
    cwd=str(AURA_ROOT),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=server_env,
    text=True
)

print(f"Uvicorn process started (PID: {server_process.pid})")
print(f"Logging to: {log_file_path}")
print("Waiting for lifespan startup (loading MMDiT & DWPose weights into GPU)...\n")


In [ ]:
# Step 7: Verify /health and /ready Checks
import time
import urllib.request
import json

# Poll /health (up to 30s)
service_up = False
for attempt in range(30):
    if server_process.poll() is not None:
        if log_file_path.exists():
            print(f"\n--- Service Process Log Dump ({log_file_path}) ---")
            print(log_file_path.read_text(encoding='utf-8', errors='ignore'))
        raise RuntimeError(f"Server process terminated unexpectedly with code {server_process.returncode}! Check {log_file_path}")
    try:
        req = urllib.request.Request("http://127.0.0.1:8001/health")
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                health_data = json.loads(resp.read().decode("utf-8"))
                print(f"Health Check: {health_data}")
                service_up = True
                break
    except Exception:
        time.sleep(1)

if not service_up:
    if log_file_path.exists():
        print(f"\n--- Service Process Log Dump ({log_file_path}) ---")
        print(log_file_path.read_text(encoding='utf-8', errors='ignore'))
    raise TimeoutError("Server failed to respond to /health within 30 seconds.")

# Poll /ready (up to 90s - model loading into GPU memory takes 15-30s)
service_ready = False
print("Polling /ready endpoint while MMDiT and DWPose load onto GPU...")
for attempt in range(90):
    if server_process.poll() is not None:
        if log_file_path.exists():
            print(f"\n--- Service Process Log Dump ({log_file_path}) ---")
            print(log_file_path.read_text(encoding='utf-8', errors='ignore'))
        raise RuntimeError(f"Server process died during model warmup! Check {log_file_path}")
    try:
        req = urllib.request.Request("http://127.0.0.1:8001/ready")
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                ready_data = json.loads(resp.read().decode("utf-8"))
                if ready_data.get("ready"):
                    print(f"Ready Check: {ready_data} (Model & DWPose loaded into GPU)")
                    service_ready = True
                    break
                else:
                    if (attempt + 1) % 10 == 0:
                        print(f"  Waiting for model warmup... (attempt {attempt + 1}/90, ready={ready_data.get('ready')})")
    except Exception:
        pass
    time.sleep(1)

if not service_ready:
    if log_file_path.exists():
        print(f"\n--- Service Process Log Dump ({log_file_path}) ---")
        print(log_file_path.read_text(encoding='utf-8', errors='ignore'))
    raise TimeoutError("Server failed to reach /ready within 90 seconds. Inspect server logs above.")

print("\n>>> STATUS: AURA GPU VTO Service is ACTIVE and READY on http://127.0.0.1:8001 <<<")


## Kaggle Networking Architecture & Inbound Reachability

### Network Boundary Reality
- **Inside Kaggle**: The service is listening securely on `http://127.0.0.1:8001`. In-notebook tests, local job submissions, and batch evaluations communicate with the service directly.
- **Outside Kaggle (Mobile App)**: Kaggle notebook containers are isolated and do **not** assign a public IP or forward inbound ports. Direct access from the internet or mobile phone to `127.0.0.1:8001` is physically impossible without a reverse tunnel.
- **Optional Temporary Tunnel**: If permitted by your environment and policy, you can establish an authenticated, temporary reverse tunnel (e.g. via Cloudflare Tunnel or ngrok) by supplying your own auth token in Kaggle Secrets (`NGROK_AUTHTOKEN` or `CLOUDFLARE_TUNNEL_TOKEN`).
- **Temporary Scope**: Any generated tunnel URL is strictly temporary and invalidates as soon as the Kaggle notebook stops.
- **Security Rule**: NEVER hardcode or commit tunnel URLs or auth tokens into repository source control or client configurations.

In [ ]:
# Step 8: Optional Authenticated Temporary Tunnel (For Mobile App Testing)
ngrok_token = os.getenv("NGROK_AUTHTOKEN")
try:
    if not ngrok_token:
        from kaggle_secrets import UserSecretsClient
        ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception:
    pass

if ngrok_token:
    print("Starting optional authenticated ngrok tunnel...")
    !pip -q install pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(8001, "http")
    print(f"Temporary Tunnel Active: {tunnel.public_url}")
    print("\nTo test with the mobile app:")
    print(f"Set in your local mobile .env: EXPO_PUBLIC_VTO_API_URL={tunnel.public_url}")
    print("Remember to revert EXPO_PUBLIC_VTO_API_URL when your Kaggle session finishes.")
else:
    print("No tunnel configured. Service is running in local-only mode (127.0.0.1:8001).")
    print("Mobile app integration from external devices remains blocked by Kaggle network boundary.")


In [ ]:
# Step 9: Local Service Smoke Test
# Verify unauthenticated request to /v1/vto/jobs is rejected with HTTP 401
import json
import urllib.request
import urllib.error

print("--- Local Route Verification ---")

# Syntactically valid request body conforming to JobCreate schema
valid_payload = {
    "category": "tops",
    "garment_id": "garment_smoke_test_12345",
    "person_input_storage_key": "test_user/inputs/person.png",
    "garment_input_storage_key": "test_user/inputs/garment.png",
    "outfit_name": "Smoke Test Outfit",
    "idempotency_key": "smoke_test_idempotency_1234567890"
}

payload_bytes = json.dumps(valid_payload).encode("utf-8")
headers = {
    "Content-Type": "application/json"
}

print("Sending valid unauthenticated POST /v1/vto/jobs with Content-Type: application/json...")
try:
    req = urllib.request.Request(
        "http://127.0.0.1:8001/v1/vto/jobs",
        data=payload_bytes,
        headers=headers,
        method="POST"
    )
    urllib.request.urlopen(req)
    # Fail closed if unauthenticated request was accepted
    raise AssertionError("SECURITY VIOLATION: Unauthenticated request was accepted by /v1/vto/jobs! Expected HTTP 401.")
except urllib.error.HTTPError as e:
    print(f"Unauthenticated request rejected as expected: HTTP {e.code} ({e.reason})")
    error_body = e.read().decode("utf-8", errors="ignore")
    print(f"Server response detail: {error_body}")
    assert e.code == 401, f"Expected HTTP 401 Unauthorized, but got HTTP {e.code}"
    print("Verification passed: Endpoint strictly enforces authentication on valid payloads.")

print("Route verification passed.")


In [ ]:
# Step 10: Safe Service Shutdown & Cleanup
import shutil

print("--- Safe Service Shutdown ---")

# 1. Disconnect tunnel if active
if 'tunnel' in locals() and tunnel:
    try:
        from pyngrok import ngrok
        ngrok.disconnect(tunnel.public_url)
        print("Tunnel disconnected.")
    except Exception as e:
        print(f"Tunnel disconnect note: {e}")

# 2. Terminate server process
if 'server_process' in locals() and server_process:
    print(f"Terminating server process (PID: {server_process.pid})...")
    server_process.terminate()
    try:
        server_process.wait(timeout=10)
        print("Server process cleanly stopped.")
    except subprocess.TimeoutExpired:
        server_process.kill()
        print("Server process killed after timeout.")

if 'log_file' in locals() and log_file and not log_file.closed:
    log_file.close()

# 3. Clean temporary files
for tmp_dir in Path("/tmp").glob("aura-vto-*"):
    shutil.rmtree(tmp_dir, ignore_errors=True)

# 4. Clear sensitive server-only credentials from environment
for key in ["SUPABASE_SERVICE_ROLE_KEY", "VTO_JWT_SECRET"]:
    os.environ.pop(key, None)

print("Status: Controlled shutdown and environment cleanup complete.")
